# Social Contagion of Trends — Analysis

This notebook presents the full analysis pipeline: data overview, interaction graph, trend detection, propagation metrics, and virality prediction results.

In [ ]:
import json
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")

DATA_DIR = Path("../data/processed")
RESULTS_DIR = Path("../results")
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Data Overview

In [ ]:
posts = pd.read_csv(DATA_DIR / "posts.csv")
comments = pd.read_csv(DATA_DIR / "comments.csv")
print(f"Posts: {len(posts)}, Comments: {len(comments)}")
print(f"Subreddits: {posts['subreddit'].nunique()}")
print(f"Unique authors (posts): {posts['author'].nunique()}")
print(f"Unique authors (comments): {comments['author'].nunique()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
posts["subreddit"].value_counts().head(15).plot.barh(ax=axes[0], title="Posts per Subreddit (Top 15)")
comments.merge(posts[["post_id", "subreddit"]], on="post_id")["subreddit"].value_counts().head(15).plot.barh(ax=axes[1], title="Comments per Subreddit (Top 15)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "data_overview.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Interaction Graph

In [ ]:
G = nx.read_gexf(str(DATA_DIR / "interaction_graph.gexf"))
print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
print(f"Density: {nx.density(G):.6f}")

degree_seq = sorted([d for n, d in G.degree()], reverse=True)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(degree_seq, bins=50, edgecolor="black")
axes[0].set_title("Degree Distribution")
axes[0].set_xlabel("Degree")
axes[0].set_ylabel("Count")
axes[0].set_yscale("log")

centrality = nx.degree_centrality(G)
top_users = sorted(centrality.items(), key=lambda x: x[1], reverse=True)[:20]
names, vals = zip(*top_users)
axes[1].barh(range(len(names)), vals)
axes[1].set_yticks(range(len(names)))
axes[1].set_yticklabels(names, fontsize=8)
axes[1].set_title("Top 20 Users by Degree Centrality")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "graph_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Trend Detection

In [ ]:
posts_trends = pd.read_csv(DATA_DIR / "posts_with_trends.csv")
user_trends = pd.read_csv(DATA_DIR / "user_trends.csv")

trend_sizes = posts_trends.groupby(["trend_cluster", "trend_label"]).size().reset_index(name="post_count")
trend_sizes = trend_sizes.sort_values("post_count", ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(trend_sizes["trend_label"].head(15), trend_sizes["post_count"].head(15))
ax.set_xlabel("Number of Posts")
ax.set_title("Top 15 Detected Trends (by post count)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "trend_detection.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Total trends detected: {posts_trends['trend_cluster'].nunique()}")
print(f"Total user-trend assignments: {len(user_trends)}")

## 4. Propagation Metrics

In [ ]:
with open(RESULTS_DIR / "propagation_metrics.json") as f:
    prop_metrics = json.load(f)

prop_df = pd.DataFrame(prop_metrics)
print(prop_df[["trend_label", "total_users", "growth_rate_12h", "num_subreddits", "mean_post_score"]].to_string(index=False))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].scatter(prop_df["growth_rate_12h"], prop_df["total_users"])
axes[0, 0].set_xlabel("Growth Rate (users/hr, first 12h)")
axes[0, 0].set_ylabel("Total Users")
axes[0, 0].set_title("Growth Rate vs Total Reach")

axes[0, 1].scatter(prop_df["mean_degree_centrality_early"], prop_df["total_users"])
axes[0, 1].set_xlabel("Mean Degree Centrality (Early Adopters)")
axes[0, 1].set_ylabel("Total Users")
axes[0, 1].set_title("Early Adopter Centrality vs Total Reach")

axes[1, 0].scatter(prop_df["num_subreddits"], prop_df["total_users"])
axes[1, 0].set_xlabel("Number of Subreddits")
axes[1, 0].set_ylabel("Total Users")
axes[1, 0].set_title("Cross-Subreddit Spread vs Total Reach")

axes[1, 1].scatter(prop_df["mean_comment_depth"], prop_df["total_users"])
axes[1, 1].set_xlabel("Mean Comment Depth")
axes[1, 1].set_ylabel("Total Users")
axes[1, 1].set_title("Discussion Depth vs Total Reach")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "propagation_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Virality Prediction Results

In [ ]:
with open(RESULTS_DIR / "metrics.json") as f:
    model_results = json.load(f)

for model_name, results in model_results.items():
    print(f"\n{'='*50}")
    print(f"Model: {model_name}")
    print(f"  Accuracy:  {results['mean_accuracy']:.4f}")
    print(f"  Precision: {results['mean_precision']:.4f}")
    print(f"  Recall:    {results['mean_recall']:.4f}")
    print(f"  F1 Score:  {results['mean_f1']:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (model_name, results) in zip(axes, model_results.items()):
    imp = results["feature_importance"]
    features = list(imp.keys())
    values = list(imp.values())
    ax.barh(features, values)
    ax.set_title(f"Feature Importance — {model_name}")
    ax.set_xlabel("Importance")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "model_results.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Conclusions

Key findings from the analysis:
1. **Trend propagation speed** — Which features best predict viral trends?
2. **Network effects** — Do well-connected early adopters accelerate spread?
3. **Cross-subreddit spread** — Do trends that appear in multiple communities grow faster?

These findings support/challenge the hypothesis that social network structure plays a significant role in trend adoption on Reddit.